Url Connection & Data Cleaning

In [1]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv
import openmeteo_requests
import requests_cache
from urllib3.util import Retry
from retry import retry
from requests.adapters import HTTPAdapter

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 5

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
    master_df = pd.concat(dataframe_list, ignore_index=True)
    master_df = master_df.drop_duplicates(subset=['latitude', 'longitude', 'bright_ti4', 'acq_date', 'acq_time'])
    print("\nHigh-capacity local aggregation complete!")
    print(f"Combined Dataset Matrix Size: {len(master_df)} Unique Operational Fire Observations")
else:
    print("\nDataset compilation failed. Check your network or API permissions.")
    raise SystemExit

Successfully added 285 fire points from Amazon Rainforest.
Successfully added 399 fire points from California.
Successfully added 14648 fire points from Siberian Taiga.
Successfully added 557 fire points from New South Wales.
Successfully added 1885 fire points from Pantanal.
Successfully added 146 fire points from Alberta.
Successfully added 4607 fire points from Mediterranean Basin.
Successfully added 3842 fire points from Indonesia.
Successfully added 156 fire points from Greece.
Successfully added 1948 fire points from Algeria.

High-capacity local aggregation complete!
Combined Dataset Matrix Size: 26857 Unique Operational Fire Observations


Data Cleaning

In [2]:
master_df.isnull().values.any() # No null values
master_df.duplicated().any() # No duplicated values
master_df.isna().sum() # No na values

latitude         0
longitude        0
bright_ti4       0
scan             0
track            0
acq_date         0
acq_time         0
satellite        0
instrument       0
confidence       0
version          0
bright_ti5       0
frp              0
daynight         0
location_name    0
dtype: int64

Data Exploration

In [3]:
variance_profile = master_df.groupby('location_name')[['frp', 'bright_ti5', 'bright_ti4']].agg(['mean','std'])
print("\n")
print(variance_profile)
print("\n")
prediction_features = master_df[['frp', 'bright_ti5', 'bright_ti4', 'track', 'scan']]
print(prediction_features.head())



                           frp             bright_ti5             bright_ti4  \
                          mean        std        mean        std        mean   
location_name                                                                  
Alberta               3.312397   3.709351  287.724110   8.361563  315.386507   
Algeria               2.854918   3.851730  299.574795  10.258323  324.215861   
Amazon Rainforest     6.762667   7.787245  293.597825   5.580146  332.407649   
California           22.085439  53.872552  301.926566  19.982351  328.776667   
Indonesia             5.874313   7.077079  293.973108   7.350372  330.036361   
Mediterranean Basin   7.299605  15.065841  300.279069  11.802327  327.297740   
New South Wales       4.800072   7.430602  284.098779   8.480838  320.954865   
Pantanal              9.389899  19.062381  291.692276   7.904777  327.397257   
Siberian Taiga        9.641443  24.274256  288.690311  10.077897  327.190782   

                                
    

Creating Risk Level Column

In [4]:
def assign_risk_level(frp_value):
  if frp_value < 5.0:
    return 0
  elif frp_value < 15.0:
    return 1
  elif frp_value < 40.0:
    return 2
  else:
    return 3

master_df['risk_level'] = master_df['frp'].apply(assign_risk_level)

master_df['is_daytime'] = master_df['daynight'].map({'D': 1, 'N': 0})

if master_df['confidence'].dtype == 'O':
    master_df['confidence_num'] = master_df['confidence'].map({'low': 0, 'nominal': 1, 'high': 2})
else:
    master_df['confidence_num'] = master_df['confidence']

# Feature Engineering
master_df['pixel_area'] = master_df['scan'] * master_df['track']
master_df['frp_density'] = master_df['frp'] / master_df['pixel_area']
master_df['thermal_diff'] = master_df['bright_ti4'] / master_df['bright_ti5']
master_df['thermal_ratio'] = master_df['bright_ti4'] / master_df['bright_ti5']
master_df['is_daytime'] = master_df['is_daytime'].fillna(0).astype(int)
master_df['confidence_num'] = pd.to_numeric(master_df['confidence_num'], errors='coerce').fillna(0).astype(float)


Prediction: Fire Radiative Power & Bright Fire Spots

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

X = master_df[['track', 'scan', 'bright_ti5', 'bright_ti4', 'latitude', 'longitude', 'is_daytime', 'confidence_num', 'pixel_area', 'thermal_diff', 'thermal_ratio']]
y = master_df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)

# Printing the shapes
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}, y_test: {y_test.shape}")


# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=120, criterion="log_loss", random_state=42)
rf_model.fit(X_train, y_train, sample_weight=None)
rf_predictions = rf_model.predict(X_test)

# Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(n_estimators=100, random_state = 42, verbose=0)
gb_model.fit(X_train, y_train, sample_weight=None)
gb_predictions = gb_model.predict(X_test)
# Add a new Dataframe to see the results clearly
test_results_df = X_test.copy()
test_results_df['True_Risk_Level (Random Forest)'] = y_test
test_results_df['Predicted_Risk_level (Random Forest)'] = rf_predictions
test_results_df['True_Risk_Level (Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (Gradient Boosting)'] = gb_predictions

test_results_df.head(50)

X_train: (21485, 11), X_test: (5372, 11), y_train: (21485,), y_test: (5372,)


,track,scan,bright_ti5,bright_ti4,latitude,longitude,is_daytime,confidence_num,pixel_area,thermal_diff,thermal_ratio,True_Risk_Level (Random Forest),Predicted_Risk_level (Random Forest),True_Risk_Level (Gradient Boosting),Predicted_Risk_Level (Gradient Boosting)
13462,0.60,0.40,290.26,305.71,55.28318,61.46269,0,0.0,0.2400,1.053228,1.053228,0,0,0,0
3885,0.38,0.42,281.09,313.86,59.24145,86.66666,0,0.0,0.1596,1.116582,1.116582,0,0,0,0
18197,0.37,0.40,293.15,304.35,36.90459,14.72761,0,0.0,0.1480,1.038206,1.038206,0,0,0,0
2760,0.36,0.38,289.17,337.50,57.68349,106.08395,0,0.0,0.1368,1.167134,1.167134,1,0,1,0
1805,0.38,0.43,295.21,348.71,59.17389,84.16926,1,0.0,0.1634,1.181227,1.181227,1,1,1,1
3789,0.38,0.42,277.97,307.63,59.17019,86.58051,0,0.0,0.1596,1.106702,1.106702,0,0,0,0
19338,0.36,0.39,294.13,310.10,41.47491,-1.48934,0,0.0,0.1404,1.054296,1.054296,0,0,0,0
4404,0.43,0.57,300.71,344.99,59.48095,124.67195,1,0.0,0.2451,1.147252,1.147252,1,2,1,2
10545,0.37,0.40,307.65,353.65,59.88600,90.12277,1,0.0,0.1480,1.149521,1.149521,2,2,2,2
16290,0.58,0.37,295.29,367.00,-18.34175,-59.05877,1,0.0,0.2146,1.242846,1.242846,1,1,1,1


Accuracy Score: How good were the model's guesses?

In [6]:
from sklearn.metrics import accuracy_score as acc
from sklearn.metrics import precision_score as pre
from sklearn.metrics import recall_score as rc
from sklearn.metrics import f1_score as f1
from sklearn.metrics import classification_report as cl_report
rf_acc_score = acc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'])
rf_pre_score = pre(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_rec_score = rc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_f1_score = f1(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')

gb_acc_score = acc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'])
gb_pre_score = pre( test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_rec_score = rc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_f1_score = f1(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
print(f" Accuracy Score: Random Forest --> {rf_acc_score}\n Precision Score: Random Forest --> {rf_pre_score}\n Recall Score: Random Forest --> {rf_rec_score}\n F1 Score: Random Forest --> {rf_f1_score}")
print("\n")
print(f" Accuracy Score: Gradient Boosting --> {gb_acc_score}\n Precision Score: Gradient Boosting --> {gb_pre_score}\n Recall Score: Gradient Boosting --> {gb_rec_score}\n F1 Score: Gradient Boosting --> {gb_f1_score}")
print(cl_report(
    test_results_df['True_Risk_Level (Gradient Boosting)'], 
    test_results_df['Predicted_Risk_Level (Gradient Boosting)'], 
    target_names=['Low Risk (0)', 'Moderate Risk (1)', 'High Risk (2)', 'Extreme Risk (3)']
))


 Accuracy Score: Random Forest --> 0.7728965003723008
 Precision Score: Random Forest --> 0.7664335937140516
 Recall Score: Random Forest --> 0.7728965003723008
 F1 Score: Random Forest --> 0.7647045684786834


 Accuracy Score: Gradient Boosting --> 0.7680565897244974
 Precision Score: Gradient Boosting --> 0.763368620680434
 Recall Score: Gradient Boosting --> 0.7680565897244974
 F1 Score: Gradient Boosting --> 0.7566049265671784
                   precision    recall  f1-score   support

     Low Risk (0)       0.86      0.90      0.88      3254
Moderate Risk (1)       0.61      0.69      0.64      1490
    High Risk (2)       0.54      0.27      0.36       471
 Extreme Risk (3)       0.89      0.31      0.45       157

         accuracy                           0.77      5372
        macro avg       0.72      0.54      0.58      5372
     weighted avg       0.76      0.77      0.76      5372



Prediction: Only 83-85% F1 Score, let's test XG Gradient Boosting

In [9]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
import numpy as np

custom_weights = {0: 1.0, 1: 2.0, 2: 4.0, 3: 7.0} 
train_sample_weights = y_train.map(custom_weights)

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    learning_rate=0.15,     
    max_depth=7,            
    min_child_weight=1,   
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)

xgb_model.fit(X_train, y_train, sample_weight=train_sample_weights)
xgb_preds = xgb_model.predict(X_test)

test_results_df['True_Risk_Level (XG Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'] = xgb_preds

xgb_acc_score = acc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'])
xgb_pre_score = pre(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_rec_score = rc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_f1_score = f1(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')

print(f" Accuracy Score: XG Gradient Boosting --> {xgb_acc_score}\n Precision Score: XG Gradient Boosting--> {xgb_pre_score}\n Recall Score: XG Gradient Boosting --> {xgb_rec_score}\n F1 Score: XG Gradient Boosting --> {xgb_f1_score}")
print(classification_report(y_test, xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

xgb_param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42
)

xgb_grid = GridSearchCV(
    estimator=xgb_base,         
    param_grid=xgb_param_grid,   
    scoring='precision_weighted', 
    cv=5
)

xgb_grid.fit(X_train, y_train, sample_weight=train_sample_weights)
best_xgb = xgb_grid.best_estimator_
print(f"\nWinning XGBoost Settings: {xgb_grid.best_params_}")

tuned_xgb_preds = best_xgb.predict(X_test)
print("\nTUNED XGBOOST SCORECARD:")
print(classification_report(y_test, tuned_xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

 Accuracy Score: XG Gradient Boosting --> 0.7697319434102755
 Precision Score: XG Gradient Boosting--> 0.7819188341565279
 Recall Score: XG Gradient Boosting --> 0.7697319434102755
 F1 Score: XG Gradient Boosting --> 0.7732578267821669
              precision    recall  f1-score   support

         Low       0.91      0.85      0.88      3254
    Moderate       0.61      0.73      0.66      1490
        High       0.51      0.46      0.49       471
     Extreme       0.60      0.48      0.54       157

    accuracy                           0.77      5372
   macro avg       0.66      0.63      0.64      5372
weighted avg       0.78      0.77      0.77      5372


Winning XGBoost Settings: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 8, 'subsample': 0.8}

TUNED XGBOOST SCORECARD:
              precision    recall  f1-score   support

         Low       0.91      0.85      0.88      3254
    Moderate       0.61      0.74      0.67      1490
        High       0.54      0.

Using JobLib to post the results and cache into memory back to the client

In [ ]:
import joblib
from joblib import Memory

joblib.dump(best_xgb, "best_xgb.pkl")
print("Model was saved as a physical file...")
cache_dir = "./wildfire_predictions"
memory = Memory(cache_dir, verbose=1)

@memory.cache
def predict_results(X_matrix):
  print("Computing predictions from user coordinates...")
  return best_xgb.predict(X_matrix)

preds_first = predict_results(X_test)
preds_second = predict_results(X_test)


Model was saved as a physical file...
________________________________________________________________________________
[Memory] Calling __main__--var-folders-c9-_k9zdr3d2958dzv4qjrpmj540000gn-T-ipykernel-3814925403.predict_results...
predict_results(       track  scan  bright_ti5  bright_ti4  latitude  longitude  is_daytime  \
10291   0.45  0.41      287.37      298.31  28.89169    9.96962           0   
6618    0.37  0.40      280.38      308.50  43.37751   -6.97089           0   
4441    0.47  0.46      290.32      308.85 -16.52453  -59.79142           0   
7500    0.37  0.41      313.24      339.29  35.15380   -3.01133           1   
9242    0.53  0.60      292.69      325.02   3.86249   96.44469           1   
6442    0.57  0.36      287.97      306.89  36.76297    7.30639           0   
7808    0.37  0.41      292.49      306.08  32.92993    3.23607           0   
761     0.47  0.45      285.10      332.38  66.73126   80.42147     ...)
Computing predictions from user coordinates..